# 工程技巧总结

# DAPO

1. 动态采样	所有 prompt 都参与训练，浪费算力	过滤掉模型已经答对的 prompt	训练效率提升 2-3x
2. Token 级损失让 GRPO 能够区分"哪些 token 是好的、哪些是坏的"，实现更精细的信用分配。
3. Dr.GRPO 的修正极其简单——只减去均值，不除以 std：

r
~
  
i
Dr.GRPO
​
 =r 
i
​
 −mean(r 
1
​
 ,…,r 
G
​
 )
 这一改动让训练后期的回答长度膨胀问题显著缓解，reward hacking 行为减少。Qwen 系列在内部训练中采纳了类似的修正。
 因为长度偏差：当一个 prompt 的所有回答 reward 方差很大（部分对部分错），除以 std 后优势被压缩；当方差很小（全对或全错），std 接近零，优势会被放大到不合理的量级。模型由此学到"产生差异化输出比答对更重要"。
Reward hacking 的温床：除以 std 等价于鼓励模型增加组内 reward 方差，而增加方差最简单的方式就是让一部分回答变得更长（更多 token、更多 chances 答对）。这是 R1-Zero 训练后期回答长度爆炸的直接原因之一。

# 多层奖励并且衡量奖励尺度 

下面代码就是rule 用来衡量明显的规则奖励， 然后 test 用来衡量内部的测试，

In [ ]:
# Rule + Test + Verifier 的三层结构
# 对于代码任务，纯单元测试 reward 不够鲁棒——模型可能写出"只通过测试用例但不通用"的硬编码答案。RTV（Rule-Test-Verifier） 是一种三层奖励：


def rtv_reward(prompt, code, test_cases):
    # Layer 1: Rule reward - 检查代码格式、长度、是否包含 forbidden pattern
    rule_score = check_format(code) + check_no_hardcode(code)
    
    # Layer 2: Test reward - 运行公开测试用例
    test_score = run_tests(code, test_cases["public"])
    
    # Layer 3: Verifier reward - 运行隐藏测试 + LLM judge 评分
    hidden_score = run_tests(code, test_cases["hidden"])
    judge_score = llm_judge(prompt, code, rubric="correctness, style, efficiency")
    
    return 0.1 * rule_score + 0.5 * test_score + 0.3 * hidden_score + 0.1 * judge_score

奖励尺度的归一化
混合多种 reward 时最大的工程问题是尺度不一致。数学题 reward 是 {0,1}，代码题通过率是 [0,1]，GenRM 分数可能是 [−3,3]，length penalty 是 [−0.5,0.5]。直接相加会让大尺度 reward 主导梯度。

ERNIE 4.5 的 Unified Rewarding System 给出标准做法——按任务域做 z-score 归一化：

r
~
  
domain
​
 = 
σ 
domain
​
 
r−μ 
domain
​
 
​
 

其中 μ 
domain
​
 ,σ 
domain
​
  是当前 batch 内同域 reward 的均值和标准差。归一化后所有 reward 都在 [−3,3] 量级，可以安全相加。

另一种做法是 GRPO 的组内归一化——同一 prompt 的 G 个 rollout 内部做 z-score。这天然消除了跨 prompt 的尺度差异，是 GRPO 相比 PPO 的一个隐含优势。